## NOTEBOOK FOR INITIAL TESTING BEFORE COPYING TO SRC PYTHON SCRIPTS 

In [ ]:
import argparse
import os
import shutil
from pathlib import Path
from langchain_docling import DoclingLoader
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer
from langchain_core.documents import Document
import sys

# Notebooks live in notebooks/, so make src/ importable (config.py, get_embedding_function.py)
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import DEFAULT_CORPUS, get_chroma_path, get_data_path
from get_embedding_function import get_embedding_function
from langchain_chroma import Chroma

In [ ]:
CORPUS = DEFAULT_CORPUS   # folder name under data/, e.g. "mash" or "hh"
DATA_PATH = get_data_path(CORPUS)            # <project>/data/<corpus>
CHROMA_PATH = str(get_chroma_path(CORPUS))   # <project>/vector_stores/chroma_<corpus>
TOKENIZER_ID = "sentence-transformers/all-MiniLM-L6-v2"
MAX_TOKENS = 512 
#higher token numbers = larger chunks but greater surrounding context? Small token numbers = smaller chunks but fragmented.
#could change to smaller chunks + retrieve neighbouring chunks later (implement later)

# Document loader and chunker 

- NOTE: Unlike PyPDF approach which loads documents first, and then uses textsplitter, docling integrates chunking 
- ! Docling stores metadata differently - will need to consider this for later

In [ ]:
# Tokenizer + max size used by the chunker. Change these to match your embedding model.

def load_documents():
    pdf_paths = [str(p) for p in Path(DATA_PATH).glob("*.pdf")]
    chunker = HybridChunker(
        tokenizer=HuggingFaceTokenizer(
            tokenizer=AutoTokenizer.from_pretrained(TOKENIZER_ID),
            max_tokens=MAX_TOKENS,
        )
    )
    loader = DoclingLoader(file_path=pdf_paths, chunker=chunker)
    chunks = loader.load()
    print(f"Loaded: {len(chunks)} chunks from {len(pdf_paths)} PDFs")
    return chunks

chunks_list = load_documents()

In [ ]:
# Alternative simpler loader (prev approach): Docling's default chunker (no custom tokenizer / HybridChunker setup).
# Use with: chunks = load_documents_alt()
# defaults to 256 tokens
def load_documents_alt():
    pdf_paths = [str(p) for p in Path(DATA_PATH).glob("*.pdf")]
    loader = DoclingLoader(file_path=pdf_paths)
    chunks = loader.load()
    print(f"Loaded: {len(chunks)} documents from  {len(pdf_paths)} PDFs")
    return chunks


In [ ]:
print(chunks_list[10].page_content)
print(chunks_list[10].metadata)

# each item in chunks contains the text content of each pdf 

# Build a vector database (chroma) that can be updated with new documents
- Need to build the vector db
- Need to allow updates if the data store is changed 

In [ ]:
# chunk may span multiple pages 
# docling has a nested metadata structure, weirder to work with 
# extraction from metadata differs from tutorial, claude says metadata requires flattening for chroma storage
chunks_list[1].metadata["dl_meta"]["doc_items"]

In [ ]:
def get_pages(chunk):
    # A chunk can span pages; return all page numbers from its Docling doc_items
    items = chunk.metadata["dl_meta"]["doc_items"]
    return sorted({p["page_no"] for item in items for p in item["prov"]})


def calculate_chunk_ids(chunks):
    # This will create IDs like "paper.pdf:3:2"
    # Source : Page Number (first page of the chunk) : Chunk Index on that page

    last_page_id = None
    current_chunk_index = 0

    for i, chunk in enumerate(chunks): # for every chunk in the list, build the unique id 
        source = Path(chunk.metadata.get("source")).name  # filename only
        page = get_pages(chunk)[0]
        current_page_id = f"{source}:{page}"

        # If the page ID is the same as the last one, increment the index.
        if current_page_id == last_page_id: #!!! CHECK?? WHAT DOES THIS DO????? WHY? 
            current_chunk_index += 1
        else:
            current_chunk_index = 0

        # Calculate the chunk ID.
        chunk_id = f"{current_page_id}:{current_chunk_index}"
        last_page_id = current_page_id

        # Add it to the chunk metadata.
        chunk.metadata["page"] = page
        chunk.metadata["id"] = chunk_id # assigns page and full id to metadata 

        # inspect: show the first few IDs
        if i < 5:
            print(f"chunk {i}: pages={get_pages(chunk)} -> id={chunk_id}")

    print(f"Assigned IDs to {len(chunks)} chunks; unique IDs: {len({c.metadata['id'] for c in chunks})}")
    return chunks


- Each page contains multiple chunks
- The first chunk on a page has chunk index 0
- Chunk index resets to 0 on each new page
- Chunks spanning multiple pages take the first page within their id 

e.g. paper.pdf:1:0, paper.pdf:1:1, paper.pdf:2:0

In [ ]:
# updating an existing database
# leverage the metadata to create a unique id per chunk

def flatten_metadata(chunk):
    # Chroma only accepts flat metadata (str/int/float/bool), so drop the nested dl_meta
    # Returns a NEW Document so the original chunk keeps its full metadata
    flat = Document(
        page_content=chunk.page_content,
        metadata={
            "source": Path(chunk.metadata["source"]).name,  # filename only, no machine path
            "page": chunk.metadata["page"],
            "headings": " > ".join(chunk.metadata["dl_meta"].get("headings", [])),
            "id": chunk.metadata["id"],
        },
    )
    print(f"flattened metadata for {flat.metadata['id']}: keys={list(flat.metadata)}")
    return flat


def add_to_chroma(chunks: list[Document]):
    # Load the existing database.
    db = Chroma(
        persist_directory=CHROMA_PATH, embedding_function=get_embedding_function()
    )

    print(f"Chunks received: {len(chunks)}")

    # Calculate Page IDs (also sets metadata["page"]).
    chunks_with_ids = calculate_chunk_ids(chunks)

    # Add or Update the documents.
    existing_items = db.get(include=[])  # IDs only, not the text etc. 
    existing_ids = set(existing_items["ids"])
    print(f"Number of existing documents in DB: {len(existing_ids)}")

    # Only add documents that don't exist in the DB.
    new_chunks = []
    for chunk in chunks_with_ids:
        if chunk.metadata["id"] not in existing_ids:
            new_chunks.append(chunk)

    if len(new_chunks):
        print(f"👉 Adding new documents: {len(new_chunks)}")
        # Flatten copies of the chunks we're adding; the IDs were already computed above.
        new_chunks = [flatten_metadata(c) for c in new_chunks]
        new_chunk_ids = [chunk.metadata["id"] for chunk in new_chunks]
        print(f"Example ID being added: {new_chunk_ids[0]}")
        print(f"Example flat metadata: {new_chunks[0].metadata}")
        db.add_documents(new_chunks, ids=new_chunk_ids)  # saves automatically
        print(f"DB now holds {len(db.get(include=[])['ids'])} chunks")
    else:
        print("✅ No new documents to add")


- updating chroma db: 
- call add_to_chroma:
- calls get_embeeding_function() ->loads the embedding model 
- calls calculate_chunk_ids() -> calculates the id per chunk within the list of chunks 


- ! NOTE: Only checks IDs not CONTENT to maximise speed. Editing a PDF may lead to chunks keeping the same ID with different text, and DB would not be updated. 
- ! Other changes e.g. chunking settings, embedding model, new metadata fields, docling vers. will all result in the same issue. For now, delete the chroma folder if making these changes via 'rm -rf chroma'

- Can later implement a config tag alongside chroma e.g. including embedding model name, chunk parameters etc. and trigger a rebuild upon any changes

In [ ]:
# Inspect flatten_metadata on one chunk
idx = 1

# flatten_metadata needs "page" and "id", which calculate_chunk_ids sets
chunks_list = calculate_chunk_ids(chunks_list)

print("=== BEFORE: metadata as it is ===")
print(chunks_list[idx].metadata)

flat = flatten_metadata(chunks_list[idx])

print("\n=== AFTER: flatten_metadata output ===")
print(flat.metadata)

print("\n=== original chunk still has full metadata? ===")
print("dl_meta" in chunks_list[idx].metadata)


- Check if the metadata flattening works properly using the prints provided 
- May need to actually create a folder/chroma store for this to work?
- Add elements to the metadata that you want to keep in chroma that may be useful for later
- https://medium.com/@shyamsundarmuthu/unlocking-high-precision-rag-the-role-of-metadata-enhancement-parsing-and-document-structuring-d5d17364b894 read this for processing tabular data in pdfs? may be quite useful for my purpose (extracting data and specific parameters etc.)
- check how the current pipeline handles tabular data (e.g. it seems to separate text from footnote etc.)

In [ ]:
add_to_chroma(chunks_list)